In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('rps_data.csv', sep=',')

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1021 entries, 0 to 1020
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   match_id           1021 non-null   int64  
 1   round              1021 non-null   int64  
 2   opp_match_wins     982 non-null    str    
 3   opp_match_winrate  977 non-null    float64
 4   stake              1021 non-null   float64
 5   opp_move           1021 non-null   str    
 6   my_move            1021 non-null   str    
 7   outcome            1021 non-null   str    
 8   score_me_before    1021 non-null   str    
 9   score_opp_before   1021 non-null   int64  
 10  prev_opp_move      1021 non-null   str    
 11  prev_outcome       1021 non-null   str    
 12  streak_draws       1021 non-null   str    
 13  prev2_opp_move     863 non-null    str    
 14  player_name        1 non-null      float64
dtypes: float64(3), int64(3), str(9)
memory usage: 140.3 KB


In [11]:
class AdaptiveRPSPredictor:
    def __init__(self):
        self.first_move_probs = {'К': 0.33, 'Н': 0.33, 'Б': 0.34}
        self.transitions = defaultdict(Counter)
        self.prev_opp_move = None
        
    def update(self, opp_move):
        if self.prev_opp_move is not None:
            self.transitions[self.prev_opp_move][opp_move] += 1
        self.prev_opp_move = opp_move
        
    def predict_proba(self):
        if self.prev_opp_move is None:
            total = sum(self.first_move_probs.values())
            return {k: v/total for k, v in self.first_move_probs.items()}
        else:
            counter = self.transitions[self.prev_opp_move]
            total = sum(counter.values())
            if total == 0:
                return {'К': 1/3, 'Н': 1/3, 'Б': 1/3}
            return {k: counter[k]/total for k in ['К', 'Н', 'Б']}
    
    def choose_move(self, strategy='max_ev'):
        probs = self.predict_proba()
        pK, pH, pB = probs['К'], probs['Н'], probs['Б']
        ev = {'К': pH - pB, 'Н': pB - pK, 'Б': pK - pH}
        if strategy == 'max_ev':
            return max(ev, key=ev.get)
        else:
            loss = {'К': pB, 'Н': pK, 'Б': pH}
            return min(loss, key=loss.get)

total = 0
correct = 0
for match_id, group in df.groupby('match_id'):
    predictor = AdaptiveRPSPredictor()
    group = group.sort_values('round')
    for _, row in group.iterrows():
        pred = predictor.choose_move()
        if pred == row['opp_move']:
            correct += 1
        total += 1
        predictor.update(row['opp_move'])

print(f"Марковская модель 1-го порядка: точность {correct}/{total} = {correct/total:.4f}")

Марковская модель 1-го порядка: точность 388/1020 = 0.3804
